In [ ]:
#!/usr/bin/env python3
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.5
# ===================================================================
# FASE: Paso 6. Generación de gráficos para resultados de CLASIFICACIÓN
# DESCRIPCIÓN: Generación de gráficos a partir de resultados guardados
# CLASIFICACIÓN CON 4 CATEGORÍAS: Empty(0), Low(1-20), Medium(21-35), High(36+)
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configurar matplotlib para guardar imágenes sin ventana emergente
import matplotlib
matplotlib.use('Agg')

print("="*60)
print("STEP 6: GENERATING PLOTS FROM SAVED RESULTS (AULA 1.5)")
print("="*60)

INPUT_DIR  = 'ml_results_classification_4clases'
OUTPUT_DIR = 'ml_results_classification_4clases'

# ==========================================
# 1. CARGAR RESULTADOS
# ==========================================
with open(os.path.join(INPUT_DIR, 'results.pkl'), 'rb') as f:
    results = pickle.load(f)

with open(os.path.join(INPUT_DIR, 'predictions.pkl'), 'rb') as f:
    predictions = pickle.load(f)

with open(os.path.join(INPUT_DIR, 'config.pkl'), 'rb') as f:
    config = pickle.load(f)

y_test = np.load(os.path.join(INPUT_DIR, 'y_test.npy'))
cm = np.load(os.path.join(INPUT_DIR, 'confusion_matrix.npy'))

df_summary = pd.read_excel(os.path.join(INPUT_DIR, 'models_summary.xlsx'))
df_summary = df_summary.set_index(df_summary.columns[0])

# Feature importance
try:
    with open(os.path.join('ml_normalized_grouped', 'selected_features.pkl'), 'rb') as f:
        selected_features = pickle.load(f)
    
    # Cargar importancia del mejor modelo (Random Forest)
    df_imp = pd.read_excel(os.path.join(INPUT_DIR, 'feature_importance_Random_Forest.xlsx'))
    importances = df_imp['Importance'].values
    feature_names_imp = df_imp['Feature'].values
except:
    importances = None
    feature_names_imp = None

# ==========================================
# 🔧 4 CLASES
# ==========================================
class_names = ['Empty(0)', 'Low(1-20)', 'Medium(21-35)', 'High(36+)']
class_colors = ['#bdc3c7', '#2ecc71', '#3498db', '#e74c3c']

# Ordenar por Accuracy
df_summary = df_summary.sort_values('Accuracy', ascending=False)
best = df_summary.index[0]
top3 = df_summary.index[:3].tolist()

print(f"   Loaded results for {len(results)} models")
print(f"   🥇 Best model: {best}")
print(f"   🏆 Top 3: {top3}")

# ==========================================
# 2. GRÁFICO DE BARRAS - COMPARATIVA DE MODELOS (TOP 3 destacados)
# ==========================================
fig, ax = plt.subplots(figsize=(12, 7))
metrics_to_plot = ['Accuracy', 'F1_weighted', 'F1_macro', 'CV_F1_mean']
x = np.arange(len(df_summary.index))
width = 0.2
colors_bar = ['#2c3e50', '#3498db', '#2ecc71', '#e74c3c']

for i, (metric, color) in enumerate(zip(metrics_to_plot, colors_bar)):
    values = df_summary[metric].values if metric in df_summary.columns else [0]*len(df_summary)
    offset = (i - len(metrics_to_plot)/2) * width + width/2
    bars = ax.bar(x + offset, values, width, label=metric, color=color, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                   f'{val:.2f}', ha='center', va='bottom', fontsize=6, rotation=90)

# Destacar top 3
for i, model_name in enumerate(df_summary.index):
    if model_name in top3:
        ax.get_xticklabels()[i].set_fontweight('bold')
        ax.get_xticklabels()[i].set_color('#e74c3c')

ax.set_xlabel('Models', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title(f'COMPARISON OF ALL MODELS — AULA 1.5 (Top 3 in red)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_summary.index, fontsize=9, rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=0.7, color='gray', linestyle='--', linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'all_models_comparison.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: all_models_comparison.png")

# ==========================================
# 3. GRÁFICO DE RADAR (TOP 3 modelos)
# ==========================================
fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': 'polar'})
radar_metrics = ['Accuracy', 'F1_weighted', 'F1_macro', 'F1_Empty', 'F1_Low', 'F1_Medium', 'F1_High']
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

top3_colors = ['#e74c3c', '#3498db', '#2ecc71']

for idx, (model_name, color) in enumerate(zip(top3, top3_colors)):
    values = []
    for metric in radar_metrics:
        val = df_summary.loc[model_name, metric] if metric in df_summary.columns else 0
        values.append(val)
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2.5, color=color, label=model_name, markersize=8)
    ax.fill(angles, values, alpha=0.1, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('TOP 3 MODELS — Radar Comparison (AULA 1.5)', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0), fontsize=10)
ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'radar_top3.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: radar_top3.png")

# ==========================================
# 4. HEATMAP DE MÉTRICAS
# ==========================================
fig, ax = plt.subplots(figsize=(12, 8))
heatmap_data = df_summary[['Accuracy', 'F1_weighted', 'F1_macro', 'F1_Empty', 'F1_Low', 'F1_Medium', 'F1_High', 'CV_F1_mean']]
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', 
            xticklabels=True, yticklabels=True,
            linewidths=0.5, ax=ax, annot_kws={"size": 8},
            cbar_kws={'label': 'Score'})
ax.set_title('HEATMAP: Metrics by Model (AULA 1.5)', fontsize=14, fontweight='bold')
ax.set_ylabel('Models', fontsize=12, fontweight='bold')
ax.set_xlabel('Metrics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_heatmap.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: metrics_heatmap.png")

# ==========================================
# 5. MATRIZ DE CONFUSIÓN (MEJOR MODELO POR ACCURACY)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Confusion Matrix — {best} (AULA 1.5)', fontsize=16, fontweight='bold')

# Absoluta
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 14, "fontweight": "bold"})
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title('Absolute Values', fontsize=13, fontweight='bold')

# Normalizada por fila
ax = axes[1]
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens', ax=ax,
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={"size": 14, "fontweight": "bold"})
ax.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax.set_title('Normalized by Row (Recall)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: confusion_matrix.png")

# ==========================================
# 6. MATRIZ DE CONFUSIÓN (TOP 3 MODELOS)
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('Confusion Matrices — TOP 3 Models (AULA 1.5)', fontsize=16, fontweight='bold')

for idx, model_name in enumerate(top3):
    y_pred = predictions[model_name]
    cm_model = pd.crosstab(
        pd.Series(y_test, name='Actual'),
        pd.Series(y_pred, name='Predicted')
    )
    # Asegurar que tiene todas las clases
    for cls in range(4):
        if cls not in cm_model.index:
            cm_model.loc[cls] = 0
        if cls not in cm_model.columns:
            cm_model[cls] = 0
    cm_model = cm_model.sort_index().sort_index(axis=1)
    
    ax = axes[idx]
    acc = df_summary.loc[model_name, 'Accuracy']
    sns.heatmap(cm_model.values, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={"size": 10})
    ax.set_title(f'{model_name}\nAcc: {acc*100:.1f}%', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix_top3.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: confusion_matrix_top3.png")

# ==========================================
# 7. FEATURE IMPORTANCE
# ==========================================
if importances is not None:
    n_top = min(15, len(feature_names_imp))
    top_idx = np.argsort(importances)[-n_top:]
    
    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.Blues(np.linspace(0.3, 0.9, n_top))
    bars = ax.barh(range(n_top), importances[top_idx], color=colors[::-1], edgecolor='white')
    ax.set_yticks(range(n_top))
    ax.set_yticklabels([feature_names_imp[i] for i in top_idx], fontsize=10)
    ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
    ax.set_title(f'Feature Importance — {best} (AULA 1.5)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    for bar, val in zip(bars, importances[top_idx]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'feature_importance.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print("   ✅ Saved: feature_importance.png")
else:
    print("   ⚠️  Feature importance not available")

# ==========================================
# 8. DISTRIBUCIÓN DE CLASES (4 clases)
# ==========================================
unique, counts = np.unique(y_test, return_counts=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(class_names, counts, color=class_colors, edgecolor='white', linewidth=1.5)
ax.set_xlabel('Occupation Level', fontsize=12, fontweight='bold')
ax.set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
ax.set_title('Test Set Class Distribution (AULA 1.5)', fontsize=14, fontweight='bold')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
            str(count), ha='center', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: class_distribution.png")

# ==========================================
# 9. MÉTRICAS POR CLASE (TOP 3)
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Per-Class Metrics — TOP 3 Models (AULA 1.5)', fontsize=16, fontweight='bold')

for idx, model_name in enumerate(top3):
    ax = axes[idx]
    f1_vals = [
        df_summary.loc[model_name, 'F1_Empty'],
        df_summary.loc[model_name, 'F1_Low'],
        df_summary.loc[model_name, 'F1_Medium'],
        df_summary.loc[model_name, 'F1_High'],
    ]
    bars = ax.bar(class_names, f1_vals, color=class_colors, edgecolor='white', linewidth=1.5)
    ax.set_title(f'{model_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('F1 Score', fontsize=11)
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y')
    for bar, val in zip(bars, f1_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', fontsize=10, fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'per_class_f1_top3.png'), dpi=150, bbox_inches='tight')
plt.close()
print("   ✅ Saved: per_class_f1_top3.png")

# ==========================================
# 10. RESUMEN
# ==========================================
print("\n" + "="*60)
print("FULL MODEL COMPARISON TABLE (AULA 1.5)")
print("="*60)
cols_show = ['Accuracy', 'F1_weighted', 'F1_macro', 'F1_High', 'Recall_High', 'CV_F1_mean']
print(df_summary[cols_show].to_string())

print(f"\n{'='*60}")
print("TOP 3 MODELS DETAILS")
print("="*60)
for i, model_name in enumerate(top3):
    acc = df_summary.loc[model_name, 'Accuracy']
    f1h = df_summary.loc[model_name, 'F1_High']
    rec = df_summary.loc[model_name, 'Recall_High']
    print(f"   {i+1}. {model_name}: Acc={acc*100:.1f}% | F1_High={f1h*100:.1f}% | Recall_High={rec*100:.1f}%")

print(f"\n{'='*60}")
print("PLOTS GENERATED SUCCESSFULLY (AULA 1.5)")
print("="*60)
print(f"Files saved in: {OUTPUT_DIR}/")
print("\n📊 Generated plots:")
print("   1. all_models_comparison.png")
print("   2. radar_top3.png")
print("   3. metrics_heatmap.png")
print("   4. confusion_matrix.png (best model)")
print("   5. confusion_matrix_top3.png (3 best models)")
print("   6. feature_importance.png")
print("   7. class_distribution.png")
print("   8. per_class_f1_top3.png")
print("="*60)

STEP 6: GENERATING PLOTS FROM SAVED RESULTS (AULA 1.5)
   Loaded results for 11 models
   🥇 Best model: Random Forest
   🏆 Top 3: ['Random Forest', 'Decision Tree', 'Extra Trees']
   ✅ Saved: all_models_comparison.png
   ✅ Saved: radar_top3.png
   ✅ Saved: metrics_heatmap.png
   ✅ Saved: confusion_matrix.png
   ✅ Saved: confusion_matrix_top3.png
   ✅ Saved: feature_importance.png
   ✅ Saved: class_distribution.png
   ✅ Saved: per_class_f1_top3.png

FULL MODEL COMPARISON TABLE (AULA 1.5)
                      Accuracy  F1_weighted  F1_macro  F1_High  Recall_High  CV_F1_mean
Unnamed: 0                                                                             
Random Forest           0.7091       0.7082    0.6615   0.5333       0.5714      0.7297
Decision Tree           0.6727       0.6406    0.6053   0.6250       0.7143      0.6729
Extra Trees             0.6545       0.6641    0.6128   0.4211       0.5714      0.7457
MLP (Neural Network)    0.6545       0.6352    0.5560   0.3158      